In [ ]:
%pip install -q langchain langchain-community langchain-core langchain-google-genai langchain-text-splitters pypdf faiss-cpu tiktoken pandas

## 1. Configure Gemini

In [ ]:
import getpass
import os
from pathlib import Path

import pandas as pd
import tiktoken
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHAT_MODEL = "gemini-3.6-flash"
EMBEDDING_MODEL = "gemini-embedding-2"
api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
if not api_key:
    api_key = getpass.getpass("Gemini API key: ")
os.environ["GOOGLE_API_KEY"] = api_key
llm = ChatGoogleGenerativeAI(model=CHAT_MODEL, temperature=0)

## 2. Load and inspect the book PDF

In [ ]:
PDF_PATH = Path("assignment") / "intro-to-ml-book.pdf"
if not PDF_PATH.is_file():
    raise FileNotFoundError(
        f"Book PDF not found at {PDF_PATH.resolve()}. Run from the project root "
        "or update PDF_PATH to the location of the supplied book."
    )
documents = PyPDFLoader(str(PDF_PATH)).load()
page_texts = [doc.page_content or "" for doc in documents]
page_lengths = [len(text) for text in page_texts]
empty_pages = [i + 1 for i, text in enumerate(page_texts) if not text.strip()]

display(pd.DataFrame([{
    "pages": len(documents),
    "extracted_characters": sum(page_lengths),
    "empty_pages": len(empty_pages)
}]))

## 3. Chunk, embed, and index

In [ ]:
import time

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=800,
    chunk_overlap=120,
    add_start_index=True,
)
chunks = text_splitter.split_documents(documents)
encoding = tiktoken.get_encoding("cl100k_base")
chunk_token_counts = [len(encoding.encode(doc.page_content)) for doc in chunks]
print(f"Created {len(chunks):,} chunks")
print("Example chunk metadata:", chunks[0].metadata)
print("Example chunk text:\n", chunks[0].page_content[:800])

embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)

# Batch embedding to respect the 30,000 tokens per minute free-tier limit
batch_size = 25  # Around ~10,000 tokens per batch
vector_store = None

print("Embedding chunks in rate-limited batches...")
for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i + batch_size]
    if vector_store is None:
        vector_store = FAISS.from_documents(documents=batch, embedding=embeddings)
    else:
        vector_store.add_documents(documents=batch)
    print(f"Indexed chunks {i} to {min(i + batch_size, len(chunks))}...")
    if i + batch_size < len(chunks):
        # Sleep 35 seconds to let the per-minute token quota safely replenish
        time.sleep(35)

retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})
print(f"FAISS index contains {vector_store.index.ntotal:,} vectors")

## 4. Retrieval and top-k experiment

Compare `k = 3, 5, 7` for one ML question and inspect the returned excerpts. FAISS reports squared L2 distances for this index; lower distances indicate closer vectors. The following sample also shows retrieved text with its source page.

In [ ]:
retrieval_query = "How does k-nearest neighbors classification work?"
retrieval_comparison = {}
for k in (3, 5, 7):
    matches = vector_store.similarity_search_with_score(retrieval_query, k=k)
    retrieval_comparison[k] = matches
    print(f"\n--- top {k} chunks ---")
    for rank, (doc, distance) in enumerate(matches, start=1):
        page = doc.metadata.get("page")
        page_label = page + 1 if isinstance(page, int) else "unknown"
        print(f"{rank}. page {page_label} | squared L2 distance={distance:.4f}")
        print(doc.page_content[:350].replace("\n", " "), "...")

## 5. Grounded answer generation

In [ ]:
rag_prompt = ChatPromptTemplate.from_template(
    """You are a machine-learning knowledge assistant. Answer using only the supplied book excerpts.
If the excerpts do not contain enough information, say exactly: "I couldn't find this information in the provided book excerpts."
Do not add facts from memory or make unsupported inferences. Cite supporting statements with the page label shown in the context, for example [page 12].

Book excerpts:
{context}

Question: {question}

Give a clear, concise answer followed by a short Evidence section with the cited page(s)."""
)
rag_chain = rag_prompt | llm | StrOutputParser()

def answer_question(question: str, k: int = 5) -> dict:
    """Retrieve book excerpts and generate a page-cited, grounded answer."""
    if not question or not question.strip():
        raise ValueError("Please provide a non-empty question.")
    if k < 1:
        raise ValueError("k must be at least 1.")

    retrieved_docs = retriever.invoke(question, config={"configurable": {"search_kwargs": {"k": k}}})
    context_parts = []
    source_pages = []
    for doc in retrieved_docs:
        page = doc.metadata.get("page")
        page_label = f"page {page + 1}" if isinstance(page, int) else "page unknown"
        context_parts.append(f"[{page_label}]\n{doc.page_content}")
        if page_label not in source_pages:
            source_pages.append(page_label)

    answer = rag_chain.invoke({"context": "\n\n".join(context_parts), "question": question.strip()})
    return {
        "question": question.strip(),
        "answer": answer,
        "sources": source_pages,
        "retrieved_chunks": retrieved_docs,
    }

## 6. End-to-end Q&A demonstration

In [ ]:
demo_questions = [
    "What is the difference between supervised and unsupervised learning?",
    "How does a decision tree make predictions, and what can cause it to overfit?",
    "What is cross-validation and why is it useful?",
]
demo_results = [answer_question(question, k=5) for question in demo_questions]
for result in demo_results:
    print(f"\n{'=' * 88}\nQuestion: {result['question']}\n")
    print(result["answer"])
    print("\nRetrieved source pages:", ", ".join(result["sources"]))
    print("Retrieved chunks:", len(result["retrieved_chunks"]))

print(f"\nPipeline check passed: {len(documents)} pages, {len(chunks)} chunks, {vector_store.index.ntotal} indexed vectors.")